<a href="https://colab.research.google.com/github/akalya33-m/AI-Student-Performance-Analyzer-/blob/main/new_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# CELL 1 — INSTALL LIBRARIES
# =========================================================

!pip install -q sentence-transformers faiss-cpu pypdf

In [ ]:
# =========================================================
# CELL 2 — IMPORT LIBRARIES
# =========================================================

import os
import re
import pickle
import numpy as np
import faiss

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [ ]:
# =========================================================
# CELL 3 — LOAD EMBEDDING MODEL
# =========================================================

print("Loading Embedding Model...")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model Loaded Successfully")


Loading Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded Successfully


In [ ]:
# =========================================================
# CELL 4 — UPLOAD DOCUMENT
# =========================================================

uploaded = files.upload()

file_path = list(uploaded.keys())[0]

print("Uploaded File:", file_path)

Saving HRPolicy.pdf to HRPolicy (1).pdf
Uploaded File: HRPolicy (1).pdf


In [ ]:
# =========================================================
# CELL 5 — CLEAN TEXT FUNCTION
# =========================================================

def clean_text(text):

    # Replace line breaks and tabs
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Fix spaces before punctuation
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    # Remove strange unicode spaces
    text = text.replace("\u00A0", " ")

    return text.strip()

In [ ]:
# =========================================================
# CELL 6 — READ DOCUMENT
# =========================================================

def read_document(file_path):

    full_text = ""

    # PDF FILE
    if file_path.endswith(".pdf"):

        reader = PdfReader(file_path)

        for page in reader.pages:

            extracted = page.extract_text()

            if extracted:

                cleaned = clean_text(extracted)

                full_text += cleaned + " "

    # TXT FILE
    elif file_path.endswith(".txt"):

        with open(file_path, "r", encoding="utf-8") as file:

            text = file.read()

            full_text = clean_text(text)

    else:
        raise Exception("Unsupported File Format")

    return full_text

In [ ]:
# =========================================================
# CELL 7 — EXTRACT TEXT
# =========================================================

text = read_document(file_path)

print("DOCUMENT EXTRACTED SUCCESSFULLY\n")

print(text[:1500])

DOCUMENT EXTRACTED SUCCESSFULLY

This policy handbook provides clear guidance to all employees regarding leave entitlements, payroll procedures, separation processes, disciplinary actions, and work timings. It is designed to ensure fairness, compliance with applicable laws, and transparency in workplace expectations. The policy applies to all permanent, temporary, and contractual employees of the Company unless otherwise specified in their employment contract. Work Timings and Attendance The standard working hours are from Monday to Friday, 9:00 AM to 6:00 PM, which includes a one-hour lunch break. Saturdays and Sundays are considered weekly off days unless specific business requirements necessitate otherwise. Employees must record their attendance daily through the Human Resource Management System or via an approved biometric or web-based application. Any late arrival beyond fifteen minutes requires prior notification to the reporting manager. Employees are entitled to one lunch break

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================================================
# CELL 8 — SMART CHUNKING
# =========================================================

def chunk_text(text, chunk_size=500, overlap=100):

    words = text.split()

    chunks = []

    current_chunk = []

    current_length = 0

    i = 0

    while i < len(words):

        word = words[i]

        word_length = len(word) + 1

        # Add word if within limit
        if current_length + word_length <= chunk_size:

            current_chunk.append(word)

            current_length += word_length

            i += 1

        else:

            # Save current chunk
            chunks.append(" ".join(current_chunk))

            # Create overlap
            overlap_words = current_chunk[-(overlap // 5):]

            current_chunk = overlap_words

            current_length = len(" ".join(current_chunk))

    # Add remaining chunk
    if current_chunk:

        chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
# =========================================================
# CELL 9 — CREATE CHUNKS
# =========================================================

chunks = chunk_text(text)

print("TOTAL CHUNKS:", len(chunks))

print("\n========================")
print("FIRST CHUNK")
print("========================\n")

print(chunks[0])

TOTAL CHUNKS: 9

FIRST CHUNK

This policy handbook provides clear guidance to all employees regarding leave entitlements, payroll procedures, separation processes, disciplinary actions, and work timings. It is designed to ensure fairness, compliance with applicable laws, and transparency in workplace expectations. The policy applies to all permanent, temporary, and contractual employees of the Company unless otherwise specified in their employment contract. Work Timings and Attendance The standard working hours are from


In [ ]:
# =========================================================
# CELL 10 — CREATE EMBEDDINGS
# =========================================================

print("Generating Embeddings...\n")

embeddings = model.encode(chunks)

embeddings = np.array(embeddings)

print("Embedding Shape:", embeddings.shape)

Generating Embeddings...

Embedding Shape: (9, 384)


In [ ]:
# =========================================================
# CELL 11 — VIEW STORED VECTORS
# =========================================================

print("TOTAL VECTORS:", len(embeddings))

print("\n========================")
print("FIRST VECTOR")
print("========================\n")

print(embeddings[0])

print("\nVECTOR DIMENSION:", len(embeddings[0]))


TOTAL VECTORS: 9

FIRST VECTOR

[-9.94752254e-03  7.77108073e-02  2.38270685e-03 -1.25909895e-02
  8.68324712e-02  8.81593600e-02  2.98866909e-02 -6.94883913e-02
 -1.72133110e-02 -2.82814912e-03  2.28404626e-02  3.01883444e-02
 -3.08123361e-02  1.55504141e-02  2.23746076e-02 -5.27936295e-02
  1.40562626e-02 -3.89000401e-02 -3.61241144e-03 -6.94790259e-02
  4.16870303e-02 -1.41020841e-03 -3.36271748e-02  3.59906331e-02
 -8.35864171e-02 -1.14465021e-02  1.46567132e-02  3.20403464e-02
 -2.68694647e-02 -1.84261077e-03 -6.34370297e-02 -3.20835912e-04
  5.82002699e-02 -5.84545545e-02  1.80857372e-03  2.39179544e-02
  4.55980450e-02 -3.21005397e-02 -1.58605184e-02 -2.87742484e-02
 -5.85268065e-02 -1.71104856e-02 -3.52386050e-02  5.66570461e-02
 -3.18590328e-02  4.84793372e-02  1.81277581e-02 -8.70394111e-02
 -7.84005672e-02  9.15826336e-02  7.94404745e-02  6.53990917e-03
  7.27077276e-02  1.08604804e-01  9.27319229e-02  7.86139667e-02
 -3.13441269e-04 -5.61590157e-02 -1.32940980e-02  3.202975

In [ ]:
# =========================================================
# CELL 12 — SHOW CHUNK + VECTOR
# =========================================================

for i in range(2):

    print("\n========================")
    print(f"CHUNK {i+1}")
    print("========================\n")

    print(chunks[i])

    print("\nVECTOR:\n")

    print(embeddings[i])

    print("\nVECTOR LENGTH:", len(embeddings[i]))



CHUNK 1

This policy handbook provides clear guidance to all employees regarding leave entitlements, payroll procedures, separation processes, disciplinary actions, and work timings. It is designed to ensure fairness, compliance with applicable laws, and transparency in workplace expectations. The policy applies to all permanent, temporary, and contractual employees of the Company unless otherwise specified in their employment contract. Work Timings and Attendance The standard working hours are from

VECTOR:

[-9.94752254e-03  7.77108073e-02  2.38270685e-03 -1.25909895e-02
  8.68324712e-02  8.81593600e-02  2.98866909e-02 -6.94883913e-02
 -1.72133110e-02 -2.82814912e-03  2.28404626e-02  3.01883444e-02
 -3.08123361e-02  1.55504141e-02  2.23746076e-02 -5.27936295e-02
  1.40562626e-02 -3.89000401e-02 -3.61241144e-03 -6.94790259e-02
  4.16870303e-02 -1.41020841e-03 -3.36271748e-02  3.59906331e-02
 -8.35864171e-02 -1.14465021e-02  1.46567132e-02  3.20403464e-02
 -2.68694647e-02 -1.84261077e

In [ ]:
# =========================================================
# CELL 13 — CREATE FAISS INDEX
# =========================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS INDEX CREATED SUCCESSFULLY")

FAISS INDEX CREATED SUCCESSFULLY


In [ ]:
# =========================================================
# CELL 14 — SAVE EVERYTHING LOCALLY
# =========================================================

os.makedirs("vectors", exist_ok=True)

# Save embeddings
np.save("vectors/embeddings.npy", embeddings)

# Save chunks
with open("vectors/chunks.pkl", "wb") as file:

    pickle.dump(chunks, file)

# Save FAISS index
faiss.write_index(index, "vectors/faiss.index")

print("ALL FILES SAVED SUCCESSFULLY")

ALL FILES SAVED SUCCESSFULLY


In [ ]:
# =========================================================
# CELL 15 — LOAD SAVED FILES
# =========================================================

loaded_embeddings = np.load("vectors/embeddings.npy")

with open("vectors/chunks.pkl", "rb") as file:

    loaded_chunks = pickle.load(file)

loaded_index = faiss.read_index("vectors/faiss.index")

print("FILES LOADED SUCCESSFULLY")


FILES LOADED SUCCESSFULLY


In [ ]:
def generate_answer(query, top_k=3):

    # =====================================
    # STEP 1 : QUERY EMBEDDING
    # =====================================

    query_embedding = model.encode([query])

    query_embedding = np.array(query_embedding)

    # =====================================
    # STEP 2 : SEARCH SIMILAR CHUNKS
    # =====================================

    distances, indices = loaded_index.search(query_embedding, top_k)

    # =====================================
    # STEP 3 : GET RETRIEVED CHUNKS
    # =====================================

    retrieved_chunks = []

    for idx in indices[0]:

        retrieved_chunks.append(loaded_chunks[idx])

    # =====================================
    # STEP 4 : CREATE CONTEXT
    # =====================================

    context = " ".join(retrieved_chunks)

    # =====================================
    # STEP 5 : SIMPLE ANSWER EXTRACTION
    # =====================================

    sentences = context.split(".")

    relevant_sentences = []

    query_words = query.lower().split()

    for sentence in sentences:

        sentence_lower = sentence.lower()

        # Check keyword match
        if any(word in sentence_lower for word in query_words):

            relevant_sentences.append(sentence.strip())

    # =====================================
    # STEP 6 : PRINT FINAL ANSWER
    # =====================================

    print("\n========================")
    print("FINAL ANSWER")
    print("========================\n")

    if relevant_sentences:

        final_answer = ". ".join(relevant_sentences)

        print(final_answer)

    else:

        print("No relevant answer found.")

In [ ]:
while True:

    query = input("\nAsk Question (type exit to stop): ")

    if query.lower() == "exit":
        break

    generate_answer(query)



FINAL ANSWER

This policy handbook provides clear guidance to all employees regarding leave entitlements, payroll procedures, separation processes, disciplinary actions, and work timings. The policy applies to all permanent, temporary, and contractual employees of the Company unless otherwise specified in their employment contract. Work Timings and Attendance The standard working hours are from provided it does not affect productivity. Overtime is either payable or compensated with a compensatory off, subject to prior approval from the manager. Leave Policy Employees are entitled to various types of leave. Unused leave may be carried forward up to a maximum of fifteen days, and encashment of earned leave is only permitted attendance daily through the Human Resource Management System or via an approved biometric or web-based application. Any late arrival beyond fifteen minutes requires prior notification to the reporting manager. Employees are entitled to one lunch break of sixty minut